# Reverse Mode Autodiff: Gradienten eines Mini-MLPs

Im Notebook `computing_the_gradient.ipynb` haben wir die Kettenregel rückwärts
auf skalare Rechenschritte angewendet. Jetzt verarbeiten wir **3 Eingaben** mit
**2 versteckten Neuronen** zu **1 Ausgabe**. Die Gewichte sind nun Matrizen.

Unser Ziel: verstehen, wie für **jedes einzelne Gewicht** eine Ableitung entsteht
und wie wir alle diese Ableitungen als Matrix berechnen können.

Wir gehen in kleinen Schritten vor:
1. Netz und Dimensionen festlegen.
2. Den Vorwärtspass ausrechnen.
3. Die Kettenregel vom Fehler zurück zu den Gewichten anwenden.
4. Die Ergebnisse durch numerisches Differenzieren und PyTorch prüfen.

Der Hauptteil benötigt nur **NumPy**. Der letzte Vergleich benötigt zusätzlich
**PyTorch**. Führe die Zellen von oben nach unten aus. Wir betrachten zunächst
**ein einziges Trainingsbeispiel**, keinen Batch.


## 1. Unser Netz: 3 → 2 → 1

$$
z = W_1 x + b_1, \qquad h = \operatorname{ReLU}(z), \qquad
\hat y = W_2 h + b_2.
$$

```text
x (3 Werte) → Linear Layer 1 → z (2 Werte) → ReLU → h (2 Werte)
                                                        ↓
                                                Linear Layer 2
                                                        ↓
                                                   ŷ (1 Wert)
                                                        ↓
                                              Verlust L mit Ziel y
```

Ein „Linear Layer“ berechnet hier eine **affine** Abbildung, also eine
Matrixmultiplikation plus Bias. ReLU wirkt auf jeden Eintrag einzeln:
$\operatorname{ReLU}(z_i)=\max(0,z_i)$.
Ohne diese Nichtlinearität könnten wir die beiden Layers zu einer einzigen
affinen Abbildung zusammenfassen.

Wir verwenden ausdrücklich **Spaltenvektoren**, also zweidimensionale Arrays:

| Größe | Bedeutung | Shape in NumPy |
|---|---|---|
| $x$ | 3 Eingabewerte | `(3, 1)` |
| $W_1$ | 2 Neuronen mit je 3 Gewichten | `(2, 3)` |
| $b_1, z, h$ | je 2 Werte | `(2, 1)` |
| $W_2$ | 1 Ausgabeneuron mit 2 Gewichten | `(1, 2)` |
| $b_2, \hat y$ | je 1 Wert | `(1, 1)` |

**Jede Zeile einer Gewichtsmatrix gehört zu einem Zielneuron.**
`@` bedeutet Matrixmultiplikation, `*` bedeutet elementweise Multiplikation.
`.T` transponiert eine Matrix: Zeilen werden zu Spalten.


In [1]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)

# Feste, kleine Zahlen: Wir können jeden Schritt von Hand nachrechnen.
x = np.array([[1.0],
              [2.0],
              [-1.0]])

W1 = np.array([[1.0, 0.0, -1.0],
               [-1.0, 1.0, 1.0]])
b1 = np.array([[1.0],
               [1.0]])

W2 = np.array([[2.0, -1.0]])
b2 = np.array([[0.5]])

y = 4.0  # Gewünschte Ausgabe für dieses eine Trainingsbeispiel.

for name, value in [("x", x), ("W1", W1), ("b1", b1), ("W2", W2), ("b2", b2)]:
    print(f"{name}: Shape {value.shape}\n{value}\n")


x: Shape (3, 1)
[[ 1.]
 [ 2.]
 [-1.]]

W1: Shape (2, 3)
[[ 1.  0. -1.]
 [-1.  1.  1.]]

b1: Shape (2, 1)
[[1.]
 [1.]]

W2: Shape (1, 2)
[[ 2. -1.]]

b2: Shape (1, 1)
[[0.5]]



## 2. Vorwärtspass: Werte berechnen

### Erster Linear Layer

Jedes Neuron berechnet eine gewichtete Summe der Eingaben plus Bias:

$$
z_1 = 1\cdot1 + 0\cdot2 + (-1)\cdot(-1) + 1 = 3,
$$
$$
z_2 = (-1)\cdot1 + 1\cdot2 + 1\cdot(-1) + 1 = 1.
$$

Beide Summen stecken gleichzeitig in `W1 @ x + b1`.


In [2]:
z = W1 @ x + b1
print("z =")
print(z)
print("Shape:", z.shape)


z =
[[3.]
 [1.]]
Shape: (2, 1)


### ReLU und zweiter Linear Layer

Beide Einträge von $z$ sind positiv. Deshalb ist hier $h=(3,1)^T$.
Die Ausgabe ist $\hat y=2\cdot3-1\cdot1+0{,}5=5{,}5$.


In [3]:
h = np.maximum(0.0, z)
y_hat = W2 @ h + b2

print("h =")
print(h)
print("Vorhersage ŷ =", y_hat.item())


h =
[[3.]
 [1.]]
Vorhersage ŷ = 5.5


### Welche skalare Größe leiten wir ab?

Beim Training möchten wir den **Verlust** verkleinern. Wir wählen:

$$L=\frac{1}{2}(\hat y-y)^2.$$

Der Faktor $1/2$ vereinfacht die Ableitung. Hier ist
$L=\frac12(5{,}5-4)^2=1{,}125$.

Wir berechnen zunächst $\partial L/\partial W_1$ und $\partial L/\partial W_2$.
Das sind **Verlustgradienten**. Weiter unten zeigen wir auch die Gradienten
der Ausgabe selbst, also $\partial\hat y/\partial W_1$ und $\partial\hat y/\partial W_2$.


In [4]:
error = y_hat - y
loss = 0.5 * error.item() ** 2
print("Abweichung ŷ − y =", error.item())
print("Verlust L =", loss)


Abweichung ŷ − y = 1.5
Verlust L = 1.125


## 3. Was bedeutet ein Gradient nach einer Matrix?

$L$ ist eine Zahl. Jedes Gewicht ist ebenfalls eine Zahl, die $L$ beeinflusst.
Wir ordnen die partiellen Ableitungen genauso an wie die Gewichte:

$$
\frac{\partial L}{\partial W_1} =
\begin{pmatrix}
\frac{\partial L}{\partial (W_1)_{11}} &
\frac{\partial L}{\partial (W_1)_{12}} &
\frac{\partial L}{\partial (W_1)_{13}} \\
\frac{\partial L}{\partial (W_1)_{21}} &
\frac{\partial L}{\partial (W_1)_{22}} &
\frac{\partial L}{\partial (W_1)_{23}}
\end{pmatrix}.
$$

**Der Gradient hat dieselbe Shape wie die jeweilige Gewichtsmatrix.**
Es handelt sich also weiterhin um gewöhnliche partielle Ableitungen,
nur übersichtlich in einer Matrix gesammelt.

Wie im skalaren Beispiel schreiben wir $\bar v=\partial L/\partial v$.
Im Code heißt diese Adjungierte `grad_v`. Für Vektoren und Matrizen enthält sie
die Ableitung nach jedem Eintrag. Die mathematischen Indizes beginnen bei 1,
die Python-Indizes bei 0.


## 4. Rückwärtspass, Schritt 1: vom Verlust zur Ausgabe

Wir starten mit $\bar L=\partial L/\partial L=1$.
Die Ableitung des quadratischen Verlustes ergibt:

$$\bar{\hat y}=\frac{\partial L}{\partial\hat y}=\hat y-y=1{,}5.$$


In [5]:
grad_loss = 1.0
grad_y_hat = grad_loss * (y_hat - y)
print("∂L/∂ŷ =", grad_y_hat.item())


∂L/∂ŷ = 1.5


## 5. Rückwärtspass, Schritt 2: zweiter Linear Layer

Aus $\hat y=(W_2)_{11}h_1+(W_2)_{12}h_2+b_2$ folgt einzeln:

$$\frac{\partial L}{\partial(W_2)_{1j}}=\bar{\hat y}\,h_j.$$

Als Matrix berechnen wir beide Gewichtsableitungen auf einmal:

$$\boxed{\bar W_2=\bar{\hat y}\,h^T},\qquad
\boxed{\bar b_2=\bar{\hat y}},\qquad
\boxed{\bar h=W_2^T\bar{\hat y}}.$$

Shapes: `(1, 1) @ (1, 2) → (1, 2)` für den Gewichtsgradienten und
`(2, 1) @ (1, 1) → (2, 1)` für den weitergereichten Gradienten.
Die Ableitung nach dem Bias erhält den Faktor 1.


In [6]:
grad_W2 = grad_y_hat @ h.T
grad_b2 = grad_y_hat.copy()
grad_h = W2.T @ grad_y_hat

print("∂L/∂W2 =", grad_W2)  # [[4.5, 1.5]]
print("∂L/∂b2 =", grad_b2)
print("∂L/∂h =")
print(grad_h)                 # [[3.0], [-1.5]]


∂L/∂W2 = [[4.5 1.5]]
∂L/∂b2 = [[1.5]]
∂L/∂h =
[[ 3. ]
 [-1.5]]


## 6. Rückwärtspass, Schritt 3: durch die ReLU

Für einen einzelnen Eintrag gilt:

$$\operatorname{ReLU}'(z_i)=
\begin{cases}1 & z_i>0,\\0 & z_i<0.\end{cases}$$

Bei $z_i=0$ existiert keine gewöhnliche Ableitung. Für den Code legen wir dort
wie PyTorch den Rückwärtswert 0 fest. Unser Beispiel liegt nicht an dieser Stelle.

$$\boxed{\bar z=\bar h\odot\operatorname{ReLU}'(z)}$$

$\odot$ steht für elementweise Multiplikation (`*`). Hier sind beide Neuronen
aktiv, sodass beide Gradienten unverändert weitergegeben werden.


In [7]:
relu_derivative = (z > 0).astype(float)
grad_z = grad_h * relu_derivative

print("ReLU-Ableitung =")
print(relu_derivative)
print("∂L/∂z =")
print(grad_z)


ReLU-Ableitung =
[[1.]
 [1.]]
∂L/∂z =
[[ 3. ]
 [-1.5]]


## 7. Rückwärtspass, Schritt 4: erster Linear Layer

### Erst ein einzelnes Gewicht verstehen

Neuron $i$ berechnet $z_i=\sum_{j=1}^{3}(W_1)_{ij}x_j+(b_1)_i$.
Nur $z_i$ hängt von dem Gewicht $(W_1)_{ij}$ ab. Deshalb liefert die Kettenregel:

$$\frac{\partial L}{\partial(W_1)_{ij}}
=\frac{\partial L}{\partial z_i}\frac{\partial z_i}{\partial(W_1)_{ij}}
=\bar z_i x_j.$$

Beispiel: Das Gewicht $(W_1)_{12}$ verbindet Eingabe 2 mit Neuron 1.
Seine Ableitung ist $\bar z_1 x_2=3\cdot2=6$.
Eine sehr kleine Erhöhung dieses Gewichts um $\varepsilon$ erhöht den Verlust
also näherungsweise um $6\varepsilon$, wenn alle anderen Größen fest bleiben.


In [8]:
# Alle sechs Ableitungen zunächst ausdrücklich einzeln berechnen.
grad_W1_einzeln = np.zeros_like(W1)
for i in range(2):        # Zielneuron
    for j in range(3):    # Eingabe
        grad_W1_einzeln[i, j] = grad_z[i, 0] * x[j, 0]
        print(f"∂L/∂W1[{i}, {j}] = {grad_z[i, 0]:g} · {x[j, 0]:g}"
              f" = {grad_W1_einzeln[i, j]:g}")


∂L/∂W1[0, 0] = 3 · 1 = 3
∂L/∂W1[0, 1] = 3 · 2 = 6
∂L/∂W1[0, 2] = 3 · -1 = -3
∂L/∂W1[1, 0] = -1.5 · 1 = -1.5
∂L/∂W1[1, 1] = -1.5 · 2 = -3
∂L/∂W1[1, 2] = -1.5 · -1 = 1.5


### Dieselbe Rechnung als äußeres Produkt

Wir multiplizieren einen Spaltenvektor mit einem Zeilenvektor:

$$\boxed{\bar W_1=\bar z\,x^T}
=\begin{pmatrix}3\\-1{,}5\end{pmatrix}
\begin{pmatrix}1&2&-1\end{pmatrix}
=\begin{pmatrix}3&6&-3\\-1{,}5&-3&1{,}5\end{pmatrix}.$$

Das ist ein **äußeres Produkt**: Jeder Eintrag des linken Vektors wird mit
jedem Eintrag des rechten Vektors multipliziert. Das Ergebnis ist eine Matrix,
kein Skalar. Shape: `(2, 1) @ (1, 3) → (2, 3)`.

Für den Bias gilt wieder $\bar b_1=\bar z$.


In [9]:
grad_W1 = grad_z @ x.T
grad_b1 = grad_z.copy()

print("∂L/∂W1 =")
print(grad_W1)
print("∂L/∂b1 =")
print(grad_b1)

np.testing.assert_allclose(grad_W1, grad_W1_einzeln)


∂L/∂W1 =
[[ 3.   6.  -3. ]
 [-1.5 -3.   1.5]]
∂L/∂b1 =
[[ 3. ]
 [-1.5]]


### Wo werden die Beiträge verschiedener Pfade addiert?

Jede Eingabe $x_j$ beeinflusst beide versteckten Neuronen.
Für den Eingabegradienten müssen wir deshalb beide Beiträge addieren:

$$\bar x_j=\sum_{i=1}^{2}(W_1)_{ij}\bar z_i,
\qquad \boxed{\bar x=W_1^T\bar z}.$$

Das entspricht dem `+=` aus dem skalaren Berechnungsgraphen.
Hier übernimmt die Matrixmultiplikation die Summation.
Die Eingaben werden beim üblichen Training nicht optimiert; ihr Gradient
macht aber sichtbar, wie der Rückwärtspass bis zum Anfang weitergeht.


In [10]:
grad_x = W1.T @ grad_z
print("∂L/∂x =")
print(grad_x)

# Beispiel: Die erste Eingabe erreicht den Verlust über beide Neuronen.
beitrag_1 = W1[0, 0] * grad_z[0, 0]
beitrag_2 = W1[1, 0] * grad_z[1, 0]
print(f"∂L/∂x1 = {beitrag_1:g} + {beitrag_2:g} = {grad_x[0, 0]:g}")


∂L/∂x =
[[ 4.5]
 [-1.5]
 [-4.5]]
∂L/∂x1 = 3 + 1.5 = 4.5


## 8. Die allgemeine Regel für einen Linear Layer

Für $v=Wu+b$ mit Spaltenvektoren und einem skalaren Verlust gilt:

| Gesuchte Ableitung | Rückwärtsregel | Shape |
|---|---|---|
| Gewichte | $\bar W=\bar v u^T$ | $(m,1)(1,n)=(m,n)$ |
| Bias | $\bar b=\bar v$ | $(m,1)$ |
| Eingabe | $\bar u=W^T\bar v$ | $(n,m)(m,1)=(n,1)$ |

Diese Regel haben wir zweimal angewendet. Entscheidend sind die **im
Vorwärtspass berechneten Eingaben** des jeweiligen Layers und der **von hinten
ankommende Gradient**. Die Gewichte dürfen wir erst nach dem gesamten
Rückwärtspass ändern.

Wir haben die Rückwärtsregeln bisher selbst angewendet; dies ist ein manueller
Rückwärtspass. Autodiff automatisiert genau diese Anwendung und Verkettung.
Wir benötigen dabei keine explizite große Jacobi-Matrix aller Zwischenwerte.

Bei mehreren Trainingsbeispielen werden deren Parametergradienten addiert.
Wenn der Verlust ein Mittelwert über den Batch ist, werden sie entsprechend gemittelt.


## 9. Und wenn wir direkt die Ausgabe ableiten möchten?

Im ursprünglichen skalaren Beispiel wurde die Ausgabe selbst abgeleitet.
Auch hier können wir statt $L$ einfach $\hat y$ als Ziel wählen.
Dann starten wir an der Ausgabe mit **1**, statt mit $\hat y-y$.
Alle folgenden lokalen Rückwärtsregeln bleiben gleich.


In [11]:
doutput_dy_hat = np.ones_like(y_hat)
doutput_dW2 = doutput_dy_hat @ h.T
doutput_dh = W2.T @ doutput_dy_hat
doutput_dz = doutput_dh * relu_derivative
doutput_dW1 = doutput_dz @ x.T

print("∂ŷ/∂W2 =")
print(doutput_dW2)
print("∂ŷ/∂W1 =")
print(doutput_dW1)

# Kettenregel: Verlustgradient = (ŷ − y) mal Ausgabegradient.
np.testing.assert_allclose(grad_W1, error.item() * doutput_dW1)
np.testing.assert_allclose(grad_W2, error.item() * doutput_dW2)


∂ŷ/∂W2 =
[[3. 1.]]
∂ŷ/∂W1 =
[[ 2.  4. -2.]
 [-1. -2.  1.]]


## 10. Kontrolle durch numerisches Differenzieren

Wir verändern jeweils **genau einen Eintrag** einer Gewichtsmatrix oder eines
Biasvektors und berechnen den Verlust neu. Die zentrale Differenz liefert:

$$\frac{\partial L}{\partial\theta}\approx
\frac{L(\theta+\varepsilon)-L(\theta-\varepsilon)}{2\varepsilon}.$$

Das ist eine unabhängige Kontrolle unserer Rückwärtsregeln, aber kein effizienter
Trainingsalgorithmus: Pro Parameter brauchen wir zwei Vorwärtspässe.
Außerdem ist das Ergebnis eine Näherung und hängt von der Schrittweite ab.
Unsere ReLU-Eingaben bleiben bei diesen kleinen Änderungen von 0 entfernt.


In [12]:
def berechne_verlust(W1, b1, W2, b2, x, y):
    z = W1 @ x + b1
    h = np.maximum(0.0, z)
    y_hat = W2 @ h + b2
    return 0.5 * (y_hat.item() - y) ** 2


parameter = {"W1": W1, "b1": b1, "W2": W2, "b2": b2}
gradienten = {"W1": grad_W1, "b1": grad_b1, "W2": grad_W2, "b2": grad_b2}
epsilon = 1e-6

for name, matrix in parameter.items():
    numerisch = np.zeros_like(matrix)
    for index in np.ndindex(matrix.shape):
        # Kopien verwenden: Die ursprünglichen Gewichte bleiben erhalten.
        plus = {key: value.copy() for key, value in parameter.items()}
        minus = {key: value.copy() for key, value in parameter.items()}
        plus[name][index] += epsilon
        minus[name][index] -= epsilon

        loss_plus = berechne_verlust(**plus, x=x, y=y)
        loss_minus = berechne_verlust(**minus, x=x, y=y)
        numerisch[index] = (loss_plus - loss_minus) / (2 * epsilon)

    np.testing.assert_allclose(numerisch, gradienten[name], rtol=1e-6, atol=1e-7)
    abweichung = np.max(np.abs(numerisch - gradienten[name]))
    print(f"{name}: numerische Kontrolle bestanden, maximale Abweichung {abweichung:.2e}")


W1: numerische Kontrolle bestanden, maximale Abweichung 4.19e-10
b1: numerische Kontrolle bestanden, maximale Abweichung 4.19e-10
W2: numerische Kontrolle bestanden, maximale Abweichung 2.10e-10
b2: numerische Kontrolle bestanden, maximale Abweichung 2.10e-10


## 11. Ein kleiner Lernschritt

Nun verwenden wir die Gradienten: Jeder Parameter wird entgegen seinem
Verlustgradienten verändert. Für die erste Gewichtsmatrix beispielsweise:

$$W_1^{\mathrm{neu}}=W_1-\eta\bar W_1.$$

$\eta$ ist die Lernrate. Bei einer ausreichend kleinen Lernrate sinkt hier der
Verlust. Eine beliebig große Lernrate garantiert das nicht.
Wir berechnen neue Arrays, damit die bisherigen Vergleichswerte erhalten bleiben.


In [13]:
lernrate = 0.01
W1_neu = W1 - lernrate * grad_W1
b1_neu = b1 - lernrate * grad_b1
W2_neu = W2 - lernrate * grad_W2
b2_neu = b2 - lernrate * grad_b2

loss_neu = berechne_verlust(W1_neu, b1_neu, W2_neu, b2_neu, x, y)
print(f"Verlust vorher: {loss:.6f}")
print(f"Verlust danach: {loss_neu:.6f}")
assert loss_neu < loss


Verlust vorher: 1.125000
Verlust danach: 0.334460


## 12. Vergleich mit PyTorch: zwei echte `Linear` Layers

Jetzt übernimmt PyTorch den Rückwärtspass. Wir setzen dieselben Gewichte und
Biaswerte ein. `Linear(3, 2)` speichert seine Gewichte ebenfalls in der Shape
`(2, 3)`.

Bei der **Eingabe** verwendet PyTorch jedoch die Konvention
`(Batchgröße, Anzahl Features)`. Unser einzelner Spaltenvektor `(3, 1)` wird
daher mit `.T` zu einer Zeile `(1, 3)`. Der Layer berechnet damit
`eingabe @ weight.T + bias`. Die Gewichtgradienten haben weiterhin genau die
Shapes unserer NumPy-Gewichtgradienten. Biasvektoren sind bei PyTorch eindimensional.

Die folgende Zelle ist optional und benötigt `torch` im Notebook-Kernel.
Sie baut bei jeder Ausführung ein neues Modell auf, sodass sich keine alten
Gradienten ansammeln. Im Training mit einem bestehenden Modell würde man die
Parametergradienten vor dem nächsten Rückwärtspass zurücksetzen.


In [14]:
import torch
from torch import nn

layer1 = nn.Linear(3, 2, dtype=torch.float64)
layer2 = nn.Linear(2, 1, dtype=torch.float64)

# Nur die Startparameter setzen; dabei keinen Ableitungsgraphen aufbauen.
with torch.no_grad():
    layer1.weight.copy_(torch.from_numpy(W1))
    layer1.bias.copy_(torch.from_numpy(b1[:, 0]))
    layer2.weight.copy_(torch.from_numpy(W2))
    layer2.bias.copy_(torch.from_numpy(b2[:, 0]))

# Eine Zeile = ein Trainingsbeispiel mit drei Eingabewerten.
x_torch = torch.tensor(x.T, dtype=torch.float64, requires_grad=True)
z_torch = layer1(x_torch)
h_torch = torch.relu(z_torch)
y_hat_torch = layer2(h_torch)
loss_torch = 0.5 * (y_hat_torch - y).square().sum()

# Ein Rückwärtspass berechnet alle Parametergradienten.
loss_torch.backward()

print("PyTorch-Vorhersage:", y_hat_torch.item())
print("PyTorch-Verlust:", loss_torch.item())
print("∂L/∂W1 =")
print(layer1.weight.grad)
print("∂L/∂W2 =")
print(layer2.weight.grad)

np.testing.assert_allclose(y_hat_torch.detach().numpy(), y_hat)
np.testing.assert_allclose(loss_torch.item(), loss)
for name, torch_grad, numpy_grad in [
    ("W1", layer1.weight.grad, grad_W1),
    ("b1", layer1.bias.grad, grad_b1[:, 0]),
    ("W2", layer2.weight.grad, grad_W2),
    ("b2", layer2.bias.grad, grad_b2[:, 0]),
    ("x", x_torch.grad, grad_x.T),
]:
    np.testing.assert_allclose(torch_grad.numpy(), numpy_grad)
    print(f"{name}: PyTorch und unser Rückwärtspass stimmen überein.")


PyTorch-Vorhersage: 5.5
PyTorch-Verlust: 1.125
∂L/∂W1 =
tensor([[ 3.0000,  6.0000, -3.0000],
        [-1.5000, -3.0000,  1.5000]], dtype=torch.float64)
∂L/∂W2 =
tensor([[4.5000, 1.5000]], dtype=torch.float64)
W1: PyTorch und unser Rückwärtspass stimmen überein.
b1: PyTorch und unser Rückwärtspass stimmen überein.
W2: PyTorch und unser Rückwärtspass stimmen überein.
b2: PyTorch und unser Rückwärtspass stimmen überein.
x: PyTorch und unser Rückwärtspass stimmen überein.


## 13. Zum Ausprobieren

Ändere jeweils die angegebenen Startwerte und führe das Notebook anschließend
wieder von oben aus. Sage vor dem Ausführen voraus, was passieren wird:

1. Setze den zweiten Eintrag von `b1` von `1.0` auf `-1.0`.
   Jetzt ist $z_2=-1$: Welche Zeile von $\bar W_1$ wird null?
   Was passiert mit der zweiten Spalte von $\bar W_2$?
2. Setze die dritte Eingabe von `-1.0` auf `0.0`.
   Warum wird die dritte Spalte von $\bar W_1$ null, auch wenn beide Neuronen aktiv sind?
3. Setze bei den ursprünglichen Parametern das Ziel `y` auf `5.5`.
   Warum verschwinden die Verlustgradienten, aber nicht alle Ausgabegradienten?
   Die Prüfung `assert loss_neu < loss` muss dann zu `assert loss_neu <= loss`
   geändert werden: Der Verlust ist bereits null.

Die in den Erklärungen und Codekommentaren angegebenen Zahlen gelten für die
ursprünglichen Startwerte. Nach Änderungen zeigen die berechneten Ausgaben die neuen Werte.

**Merksatz:** Für einen Linear Layer ist der Gewichtsgradient das äußere Produkt
„ankommender Gradient × Eingabe transponiert“. Jede Matrixposition enthält
weiterhin eine einzelne partielle Ableitung nach genau einem Gewicht.


## 14. Kompaktversion: alle Parametergradienten auf einen Blick

Dasselbe **3 → 2 → 1-Netz**, ein Trainingsbeispiel, Spaltenvektoren:

$$
 x \xrightarrow{W_1x+b_1} z \xrightarrow{\mathrm{ReLU}} h
 \xrightarrow{W_2h+b_2} \hat y
 \xrightarrow{\frac12(\hat y-y)^2} L.
$$

**Vorwärts** berechnen wir die Werte. **Rückwärts** wenden wir die Kettenregel an.
Dabei bezeichnet $\delta_2=\partial L/\partial\hat y$ den Gradienten an der
Ausgabe und $\delta_1=\partial L/\partial z$ den Gradienten vor der ReLU:

| Rückwärts von rechts nach links | Regel |
|---|---|
| Verlust → Ausgabe | $\delta_2=\hat y-y$ |
| Parameter des zweiten Layers | $\nabla_{W_2}L=\delta_2h^T,\quad \nabla_{b_2}L=\delta_2$ |
| Durch zweiten Layer und ReLU | $\delta_1=(W_2^T\delta_2)\odot(z>0)$ |
| Parameter des ersten Layers | $\nabla_{W_1}L=\delta_1x^T,\quad \nabla_{b_1}L=\delta_1$ |

Die Maske $(z>0)$ lässt den Gradienten nur durch aktive ReLU-Neuronen zurück.
Bei $z=0$ verwenden wir den Rückwärtswert 0. `@` ist Matrixmultiplikation,
`*` elementweise Multiplikation, `.T` transponiert.

Die folgende Zelle läuft **auch allein**, ohne vorherige Zellen und ohne PyTorch.


In [ ]:
import numpy as np

# 1. Startwerte: Jede Zeile einer Gewichtsmatrix gehört zu einem Zielneuron.
x = np.array([[1.0], [2.0], [-1.0]])           # (3, 1)
y = 4.0
W1 = np.array([[1.0, 0.0, -1.0],
               [-1.0, 1.0, 1.0]])            # (2, 3)
b1 = np.array([[1.0], [1.0]])                 # (2, 1)
W2 = np.array([[2.0, -1.0]])                  # (1, 2)
b2 = np.array([[0.5]])                       # (1, 1)

# 2. Vorwärts: Eingaben der Layers für den Rückwärtspass merken.
z = W1 @ x + b1
h = np.maximum(0.0, z)
y_hat = W2 @ h + b2
loss = 0.5 * (y_hat.item() - y) ** 2

# 3. Rückwärts: ankommender Gradient × lokale Ableitung.
delta2 = y_hat - y                           # ∂L/∂ŷ
grad_W2 = delta2 @ h.T                      # (1, 1) @ (1, 2) → (1, 2)
grad_b2 = delta2.copy()                     # Bias hat lokale Ableitung 1.
delta1 = (W2.T @ delta2) * (z > 0)           # ∂L/∂z: zurück durch Layer und ReLU
grad_W1 = delta1 @ x.T                      # (2, 1) @ (1, 3) → (2, 3)
grad_b1 = delta1.copy()

print(f"Vorhersage ŷ = {y_hat.item():g}, Verlust L = {loss:g}")
for name, gradient in [("W1", grad_W1), ("b1", grad_b1),
                       ("W2", grad_W2), ("b2", grad_b2)]:
    print(f"\n∂L/∂{name} (Shape {gradient.shape}):\n{gradient}")


### Die Idee hinter den vier Gradienten

Für ein einzelnes Gewicht gilt immer:

$$
\underbrace{\frac{\partial L}{\partial W_{ij}}}_{\text{Gewichtsgradient}}
=
\underbrace{\delta_i}_{\text{Gradient am Zielneuron}}
\cdot
\underbrace{u_j}_{\text{Eingabe an diesem Gewicht}},
\qquad
\underbrace{\frac{\partial L}{\partial b_i}}_{\text{Biasgradient}}=\delta_i.
$$

Hier ist $u$ die Eingabe des jeweiligen Layers, also $x$ oder $h$.
Das äußere Produkt `delta @ eingabe.T` berechnet alle Gewichtsableitungen gleichzeitig.
Der Bias wird einfach addiert; seine lokale Ableitung ist deshalb 1.

**Mit unseren Zahlen:** $\delta_2=1{,}5$ und $\delta_1=(3,-1{,}5)^T$.
Daraus entstehen:

$$
\nabla_{W_1}L=\begin{pmatrix}3&6&-3\\-1{,}5&-3&1{,}5\end{pmatrix},
\quad \nabla_{b_1}L=\begin{pmatrix}3\\-1{,}5\end{pmatrix},
\quad \nabla_{W_2}L=\begin{pmatrix}4{,}5&1{,}5\end{pmatrix},
\quad \nabla_{b_2}L=\begin{pmatrix}1{,}5\end{pmatrix}.
$$

Beispiel: $\partial L/\partial(W_1)_{12}=3\cdot2=6$.
Jeder Gradient hat dieselbe Form wie sein Parameter.
Erst **nach dem gesamten Rückwärtspass** folgt bei Bedarf der Lernschritt
`parameter = parameter - lernrate * gradient`.

**Merksatz:** Gewichtsgradient = ankommender Gradient × Eingabe transponiert;
Biasgradient = ankommender Gradient. Das gilt hier für ein einzelnes Trainingsbeispiel.
